In [ ]:
%matplotlib widget
def plot_doughnut_area_scaled(df_dn=None, file_path=None,
                              inner_domain='social', outer_domain='ecological',
                              R_soc=1.0, R_eco=1.5, scale_eco=0.008,
                              gap_frac=0.86, figsize=(10,10), title=None,
                              save_path=None, zoom=1.0,
                              # ΝΕΕΣ ΠΑΡΑΜΕΤΡΟΙ: Χάρτες Αντιστοίχισης Ετικετών
                              map_social=None, map_ecological=None):
    """Doughnut plot — no border lines, corrected inner labels, improved palettes."""
    import numpy as np
    import matplotlib.pyplot as plt
    import os
    import pandas as pd
    from matplotlib import colors as mcolors
    from collections import OrderedDict
    import textwrap as _tw

    # Προετοιμασία Χαρτών Ετικετών (με προεπιλογή σε κενό λεξικό αν δεν δοθεί)
    map_social = map_social or {}
    map_ecological = map_ecological or {}

    # load if needed
    if df_dn is None:
        if not file_path or not os.path.exists(file_path):
            raise FileNotFoundError("Provide df_dn or valid file_path")
        df_dn = pd.read_csv(file_path)
        df_dn.columns = df_dn.columns.str.strip()
        # Χρήση .get() με ασφαλή fallback σε κενή συμβολοσειρά
        df_dn['domain'] = df_dn.get('domain', '').astype(str).str.strip()

    # pick ratio column robustly
    ratio_cols = [c for c in df_dn.columns if 'ratio' in c.lower()]
    prefer = [c for c in ratio_cols if 'end' in c.lower()] + ratio_cols
    
    raw = None
    if prefer:
        raw = prefer[0]
    elif 'value' in df_dn.columns:
        raw = 'value'
    
    if raw:
        df_dn[raw] = pd.to_numeric(df_dn[raw], errors='coerce').fillna(0)
    else:
        # Δεν βρέθηκε στήλη ratio/value - δημιουργία προσωρινής για να μην σπάσει
        df_dn['__ratio_tmp'] = 0.0
        raw = '__ratio_tmp'

    vals = df_dn[raw].astype(float).values
    if vals.size == 0 and raw != '__ratio_tmp':
        raise RuntimeError("No numeric values found for ratio/value")
        
    # Scale ratios
    if vals.max() > 1.0:
        df_dn['ratio_pct'] = vals
        df_dn['ratio_scaled'] = (df_dn['ratio_pct'] / 100.0).clip(lower=0)
    else:
        df_dn['ratio_scaled'] = vals.clip(0, 1)
        df_dn['ratio_pct'] = df_dn['ratio_scaled'] * 100.0

    # split domains
    df_soc = df_dn[df_dn['domain'].str.lower() == inner_domain.lower()].copy().reset_index(drop=True)
    df_eco = df_dn[df_dn['domain'].str.lower() == outer_domain.lower()].copy().reset_index(drop=True)
    if df_soc.empty or df_eco.empty:
        raise RuntimeError("Inner or outer domain empty — check CSV / 'domain' column")

    # geometry
    N_soc, N_eco = len(df_soc), len(df_eco)
    gap = float(gap_frac)
    theta_soc = np.linspace(0.0, 2*np.pi, N_soc, endpoint=False)[::-1]
    theta_eco = np.linspace(0.0, 2*np.pi, N_eco, endpoint=False)[::-1]
    width_soc = (2*np.pi / max(1, N_soc)) * gap
    width_eco = (2*np.pi / max(1, N_eco)) * gap

    # radii and heights
    s_pct = df_soc['ratio_pct'].clip(0, 100).values
    # Note: Squared scaling for area effect based on the original code logic
    r_inner = R_soc * np.sqrt(np.maximum(0.0, 1.0 - (s_pct / 100.0))) 
    height_soc = R_soc - r_inner

    o_pct = df_eco['ratio_pct'].clip(lower=0).values
    r_outer = R_eco * np.sqrt(1.0 + (o_pct * scale_eco))
    height_eco = r_outer - R_eco

    # colors helpers
    def to_mpl_colors(col_series, n):
        if col_series is None: return None
        try:
            arr = np.array(col_series.tolist() if hasattr(col_series, "tolist") else col_series)
            if len(arr) == int(n) and all(isinstance(x, str) and x.strip().startswith('#') for x in arr):
                return arr.tolist()
        except Exception:
            pass
        return None

    def build_scaled_colors(values, light_hex, dark_hex, vmin=None, vmax=None):
        """Per-domain scale: min_positive -> light_hex, max -> dark_hex."""
        vals = np.array(values, dtype=float)
        if vals.size == 0: return []
        finite = vals[np.isfinite(vals)]
        if finite.size == 0: return []
        pos = finite[finite > 0]
        
        if pos.size > 0:
            vmin_f = float(np.nanmin(pos))
            vmax_f = float(np.nanmax(pos))
        else:
            vmin_f, vmax_f = 0.0, float(np.nanmax(finite))

        if np.isclose(vmax_f, vmin_f):
            t = np.zeros_like(vals, dtype=float)
            t[(vals > 0) & np.isfinite(vals)] = 1.0
        else:
            t_raw = (vals - vmin_f) / (vmax_f - vmin_f)
            t = np.clip(np.where(np.isfinite(t_raw), t_raw, 0.0), 0.0, 1.0)
            t = np.where(vals <= 0, 0.0, t)

        light_rgb = np.array(mcolors.to_rgb(light_hex))
        dark_rgb = np.array(mcolors.to_rgb(dark_hex))
        rgbs = (1.0 - t[:, None]) * light_rgb + t[:, None] * dark_rgb
        return [mcolors.to_hex(rgb) for rgb in rgbs]

    def _is_color_array(arr, n):
        try:
            return isinstance(arr, (list, tuple, np.ndarray)) and len(arr) == int(n) and all(isinstance(x, str) and x.strip().startswith('#') for x in arr)
        except Exception:
            return False

    # social (inner) -> Reds (shortfall), ecological (outer) -> Purples (overshoot)
    inner_from_col = to_mpl_colors(df_soc.get('color'), N_soc)
    outer_from_col = to_mpl_colors(df_eco.get('color'), N_eco)
    # explicit light/dark endpoints for interpolation
    light_hex = '#ffc6c4'  # lightest for smallest values
    dark_hex = '#672044' # darkest for largest values

    if inner_from_col is not None and _is_color_array(inner_from_col, N_soc):
        inner_colors = inner_from_col
    else:
        inner_colors = build_scaled_colors(df_soc['ratio_pct'].values, light_hex, dark_hex)

    if outer_from_col is not None and _is_color_array(outer_from_col, N_eco):
        outer_colors = outer_from_col
    else:
        outer_colors = build_scaled_colors(df_eco['ratio_pct'].values, light_hex, dark_hex)

    # prepare lookup maps (normalize keys to lower for robust matching)
    def _norm_key(k): return str(k).strip().lower()
    label_map_soc_norm = { _norm_key(k): v for k, v in map_social.items() }
    label_map_eco_norm = { _norm_key(k): v for k, v in map_ecological.items() }

    # keys for lookup
    soc_keys_raw = df_soc.get('indCode', df_soc.get('indicator', pd.Series([str(i) for i in range(N_soc)]))).astype(str).values
    eco_keys_raw = df_eco.get('indCode', df_eco.get('indicator', pd.Series([str(i) for i in range(N_eco)]))).astype(str).values

    # if indicator/long names present, prefer them
    soc_long = df_soc.get('dimension') if 'dimension' in df_soc.columns else None
    eco_long = df_eco.get('dimension') if 'dimension' in df_eco.columns else None

    # plotting: create figure+ax BEFORE any ax.* calls
    # create an independent figure + polar axis so we don't reuse the global/current figure
    fig = plt.figure(figsize=figsize, dpi=100)
    ax = fig.add_subplot(1, 1, 1, projection='polar')
    ax.set_facecolor('white')
# ...existing code...
    xs = np.linspace(0, 2*np.pi, 400)
    total_band = float(max(1e-6, R_eco - R_soc))
    # proportions: inner thin, middle thick, outer thin (adjust as desired)
    prop_inner = 0.15
    prop_middle = 0.70
    prop_outer = 0.15
    inner_thick = total_band * prop_inner
    middle_thick = total_band * prop_middle
    outer_thick = total_band * prop_outer

    # radial boundaries (from inner -> outer)
    inner_r0 = R_soc
    inner_r1 = inner_r0 + inner_thick
    middle_r0 = inner_r1
    middle_r1 = middle_r0 + middle_thick
    outer_r0 = middle_r1
    outer_r1 = R_eco  # should equal outer_r0 + outer_thick (numerical rounding OK)

    # colors: inner & outer dark green, middle lighter and thicker
    dark_green = "#227443"
    mid_green = "#6eb446" # existing green tone (slightly lighter)

    # draw three annuli (no borders)
    ax.fill_between(xs, inner_r0, inner_r1, color=dark_green, alpha=1.0, zorder=0, linewidth=0)
    ax.fill_between(xs, middle_r0, middle_r1, color=mid_green, alpha=1.0, zorder=1, linewidth=0)
    ax.fill_between(xs, outer_r0, outer_r1, color=dark_green, alpha=1.0, zorder=2, linewidth=0)

    # safe ring (visual centre band) — subtle translucent overlay
    ax.fill_between(xs, R_soc, R_eco, color='#6eb446', alpha=0.28, zorder=0, linewidth=0)

    # inner social annuli
    # build per-item label list
    if 'dimension' in df_soc.columns:
        raw_dims = df_soc['dimension'].astype(str).tolist()
    else:
        raw_dims = []
        for i, key in enumerate(soc_keys_raw):
            # ΧΡΗΣΗ ΤΗΣ ΝΕΑΣ map_social_norm
            d = label_map_soc_norm.get(_norm_key(key)) or (soc_long.iloc[i] if soc_long is not None else str(key))
            raw_dims.append(str(d))

    label_to_indices = OrderedDict()
    for idx, lab in enumerate(raw_dims):
        lab = lab.strip()
        label_to_indices.setdefault(lab, []).append(idx)

    # draw one faint wedge per distinct label covering the angular span of all its bars
    nsteps = 48  # smoother wedge edges
    for lab, idxs in label_to_indices.items():
        ang_starts = [float(theta_soc[i] - width_soc/2.0) for i in idxs]
        ang_ends   = [float(theta_soc[i] + width_soc/2.0) for i in idxs]
        ang0 = min(ang_starts)
        ang1 = max(ang_ends)
        
        a0 = ang0 % (2*np.pi)
        a1 = ang1 % (2*np.pi)
        if a1 < a0:
            th1 = np.linspace(a0, 2*np.pi, nsteps//2, endpoint=False)
            th2 = np.linspace(0.0, a1, nsteps - th1.size)
            th = np.concatenate([th1, th2])
        else:
            th = np.linspace(a0, a1, nsteps)
            
        r0_max = float(np.max(r_inner[np.array(idxs, dtype=int)]))
        try:
            wedge_col = inner_colors[idxs[0]]
        except Exception:
            wedge_col = inner_colors if isinstance(inner_colors, str) else '#ffffff'
            
        ax.fill_between(th, 0.0, r0_max, color=wedge_col, alpha=0.12, zorder=2, linewidth=0)
        
    # then draw the inner bars on top
    inner_bars = ax.bar(theta_soc, height_soc, width=width_soc, bottom=r_inner,
                        color=inner_colors, edgecolor=None, linewidth=0, zorder=3, align='center', antialiased=False)

    # outer ecological bars — same: no borders
    outer_bars = ax.bar(theta_eco, height_eco, width=width_eco, bottom=R_eco,
                        color=outer_colors, edgecolor=None, linewidth=0, zorder=4, align='center', antialiased=False)

    # curved, centered ring titles (per-character)
    def _char_arc_draw(ax, text, radius, theta_center=np.pi/2.0, arc_span=np.pi*0.8,
                      fontsize=14, color='white', zorder=7):
        txt = str(text or "")
        if not txt: return
        chars = list(txt)
        n = len(chars)
        angles = theta_center + np.linspace(arc_span/2.0, -arc_span/2.0, n)
        for ch, ang in zip(chars, angles):
            if ch == ' ': continue
            rot = np.rad2deg(ang) - 90.0
            if rot > 90: rot -= 180
            if rot < -90: rot += 180
            ax.text(ang, radius, ch,
                    rotation=rot, rotation_mode='anchor',
                    ha='center', va='center',
                    fontsize=fontsize, fontweight='bold',
                    color=color, zorder=zorder, clip_on=False)

    # compute mid-radius for outer/inner dark green rings
    outer_mid_r = outer_r0 + (outer_r1 - outer_r0) * 0.5
    inner_mid_r = inner_r0 + (inner_r1 - inner_r0) * 0.5

    # draw arc text
    _char_arc_draw(ax, "ECOLOGICAL CEILING", outer_mid_r, theta_center=np.pi/2.0, arc_span=np.pi*0.52, fontsize=10, color='white', zorder=7)
    _char_arc_draw(ax, "SOCIAL FOUNDATION", inner_mid_r, theta_center=np.pi/2.0, arc_span=np.pi*0.44, fontsize=10, color='white', zorder=7)

    # helper: stable rotation & alignment for polar text + forced 1- or 2-line label formatting
    def _text_props(mid_angle):
        deg = np.rad2deg(mid_angle)
        rot = deg - 90.0
        if rot > 90: rot -= 180
        if rot < -90: rot += 180
        ha = 'left' if np.cos(mid_angle) >= 0 else 'right'
        return rot, ha

    def two_line_label(s):
        s = str(s).strip()
        parts = s.split()
        if len(parts) <= 1: return s
        if len(parts) == 2: return parts[0] + '\n' + parts[1]
        k = len(parts) // 2
        return ' '.join(parts[:k]) + '\n' + ' '.join(parts[k:])

    # place each dimension title ONCE on the MIDDLE ring.
    middle_center_r = inner_r1 + (middle_r1 - middle_r0) * 0.5
    
    centers = []
    for lab, idxs in label_to_indices.items():
        # Circular mean of the group's span
        ang_pts = []
        for i in idxs:
            ang_pts.append(float(theta_soc[i] - width_soc/2.0))
            ang_pts.append(float(theta_soc[i] + width_soc/2.0))
        ang_pts = np.array(ang_pts)
        mean_ang = np.arctan2(np.mean(np.sin(ang_pts)), np.mean(np.cos(ang_pts)))
        if mean_ang < 0: mean_ang += 2*np.pi
        centers.append((lab, mean_ang))

    # now place the labels
    for lab, mid in centers:
        rot, ha = _text_props(mid)
        wrapped = two_line_label(lab)
        ax.text(mid, middle_center_r, wrapped, rotation=rot, rotation_mode='anchor',
                ha=ha, va='center', fontsize=12, color='white',
                zorder=7, clip_on=False)
        
    # ecological indicator labels: place just outside the outer ring
    eco_label_r = outer_r1 + (total_band * 0.08) + 0.02
    
    eco_texts = []
    for i, key in enumerate(eco_keys_raw):
        # ΧΡΗΣΗ ΤΗΣ ΝΕΑΣ map_ecological_norm
        lbl = label_map_eco_norm.get(_norm_key(key))
        if not lbl and eco_long is not None:
            try: lbl = str(eco_long.iloc[i])
            except Exception: lbl = None
        if not lbl: lbl = str(key)
        eco_texts.append(lbl.strip())

    label_to_indices_eco = OrderedDict()
    for idx, lab in enumerate(eco_texts):
        label_to_indices_eco.setdefault(lab, []).append(idx)

    offset_angle = 0.0

    for lab, idxs in label_to_indices_eco.items():
        ang_pts = []
        for i in idxs:
            ang_pts.append(float(theta_eco[i] - width_eco/2.0))
            ang_pts.append(float(theta_eco[i] + width_eco/2.0))
        ang_pts = np.array(ang_pts)
        mean_ang = np.arctan2(np.mean(np.sin(ang_pts)), np.mean(np.cos(ang_pts)))
        if mean_ang < 0: mean_ang += 2*np.pi
        mean_ang += offset_angle
        
        rot, _ = _text_props(mean_ang)
        wrapped = two_line_label(lab)
        ax.text(mean_ang, eco_label_r, wrapped, rotation=rot, ha='center', va='center', fontsize=12, zorder=6, clip_on=False)

    # optional tooltips (mplcursors)
    try:
        import mplcursors
    except ImportError:
        mplcursors = None
        
    # optional tooltips (mplcursors)
    try:
        import mplcursors
    except ImportError:
        mplcursors = None

    if mplcursors is not None:
        # build a flat list of Rectangle patches (individual bars) and attach cursor to them
        patches = []
        if 'inner_bars' in locals() and inner_bars is not None:
            patches.extend(list(inner_bars))
        if 'outer_bars' in locals() and outer_bars is not None:
            patches.extend(list(outer_bars))

        if patches:
            cursor = mplcursors.cursor(patches, hover=True)

            @cursor.connect("add")
            def on_add(sel):
                artist = sel.artist
                try:
                    idx = patches.index(artist)
                except ValueError:
                    return

                # determine which bar (inner/outer) was hovered
                n_inner = len(list(inner_bars)) if 'inner_bars' in locals() and inner_bars is not None else 0

                if idx < n_inner:
                    i = idx
                    key = soc_keys_raw[i]
                    pct = float(df_soc['ratio_pct'].iloc[i])
                    desc = label_map_soc_norm.get(_norm_key(key), str(key))
                    ax_local = inner_bars[0].axes if len(inner_bars) else artist.axes
                else:
                    i = idx - n_inner
                    key = eco_keys_raw[i]
                    pct = float(df_eco['ratio_pct'].iloc[i])
                    desc = label_map_eco_norm.get(_norm_key(key), str(key))
                    ax_local = outer_bars[0].axes if len(outer_bars) else artist.axes

                # show descriptive text
                sel.annotation.set_text(f"{desc}: {pct:.2f}%")
                sel.annotation.get_bbox_patch().set_alpha(0.95)
                sel.annotation.get_bbox_patch().set_boxstyle("round,pad=0.6")

                # hide arrow (if present) so annotation is a simple box
                try:
                    if hasattr(sel.annotation, "arrow_patch") and sel.annotation.arrow_patch is not None:
                        sel.annotation.arrow_patch.set_visible(False)
                except Exception:
                    pass

                # place the annotation at the top-center of the axes (axes-fraction coords)
                try:
                    sel.annotation.set_transform(ax_local.transAxes)
                    # position and anchor in axes fraction coordinates (x=0..1, y=0..1)
                    sel.annotation.xy = (0.5, 0.98)
                    sel.annotation.set_position((0.5, 0.98))
                    sel.annotation.set_ha('center')
                    sel.annotation.set_va('top')
                except Exception:
                    # fallback: keep default behavior if positioning fails
                    pass

                # avoid the annotation following the mouse (stick it)
                try:
                    sel.annotation.set_draggable(False)
                except Exception:
                    pass    # final styling: hide polar ticks and frame
    try:
        max_outer = float(np.max(r_outer)) if 'r_outer' in locals() or 'r_outer' in globals() else R_eco
    except Exception:
        max_outer = R_eco

    pad = 0.01
    top_needed = max(max_outer, eco_label_r) + pad # Ensures labels are visible
    
    try:
        zoom_f = float(zoom)
    except Exception:
        zoom_f = 1.0
        
    # Scale visible top limit based on zoom factor
    visible_top = R_soc + (top_needed - R_soc) * zoom_f
    visible_top = max(visible_top, R_soc + 1e-6)

    # ΑΠΛΟΠΟΙΗΜΕΝΟ STYLING - πιο καθαρό
    ax.set_ylim(0, visible_top)
    ax.set_yticklabels([])
    ax.set_xticks([])  # hide polar ticks (angles)
    ax.grid(False)
    
    # Απενεργοποίηση του πολικού frame
    if 'polar' in ax.spines:
        ax.spines['polar'].set_visible(False)
        
    ax.set_title(title, fontsize=14, y=1.06)
    plt.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')

    return fig, ax

# --- ΕΚΤΕΛΕΣΗ (Παράδειγμα) ---

# Αντικατάσταση των καθολικών μεταβλητών με πραγματικά λεξικά (ή None)
MAP_SOCIAL_EXAMPLE = {
    'S1': 'Adequate Housing',
    'S2': 'Access to Energy',
    # ...προσθέστε τις δικές σας αντιστοιχίσεις εδώ...
}

MAP_ECOLOGICAL_EXAMPLE = {
    'E1': 'Climate Change',
    'E2': 'Ocean Acidification',
    # ...προσθέστε τις δικές σας αντιστοιχίσεις εδώ...
}

file_path_global_data = globals().get('file_path_global_data',
    r"a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\9_20250516_Doughnut-GlobalTablesData.csv")

# Βεβαιωθείτε ότι το αρχείο υπάρχει ή παρέχετε ένα DataFrame
try:
    fig, ax = plot_doughnut_area_scaled(
        file_path=file_path_global_data,
        figsize=(15,15),
        zoom=0.3,
        # Πέρασμα των χαρτών ετικετών ως ορίσματα
        map_social=MAP_SOCIAL_EXAMPLE,
        map_ecological=MAP_ECOLOGICAL_EXAMPLE
    )
    plt.show()
except FileNotFoundError as e:
    print(f"Σφάλμα: Δεν βρέθηκε το αρχείο δεδομένων. {e}")
except RuntimeError as e:
    print(f"Σφάλμα κατά την επεξεργασία των δεδομένων: {e}")
except Exception as e:
    print(f"Γενικό Σφάλμα: {e}")